# Volatility-scaled trend with a fixed, pre-set target

In [ ]:
import numpy as np
import pandas as pd

def synthetic_prices():
    rng = np.random.default_rng(7)
    returns = rng.normal(0.0005, 0.012, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))


In [ ]:
VOL_TARGET = 0.10          # chosen before looking at this data
LOOKBACK = 60

def signal(prices):
    """Trend sign scaled by trailing realized vol (all lagged)."""
    rets = prices.pct_change()
    trend = np.sign(rets.rolling(LOOKBACK).mean().shift(1))
    realized_std = rets.rolling(20).std().shift(1) * np.sqrt(252)
    scale = (VOL_TARGET / realized_std).clip(upper=3.0)
    return (trend * scale).fillna(0.0)


In [ ]:
def backtest(prices, cost_bps=10):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    gross = (positions * asset_returns).mean(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(0.0)
    return gross - turnover * cost_bps / 10_000

def sharpe(pnl):
    pnl_std = pnl.std()
    return float(pnl.mean() / pnl_std * np.sqrt(252)) if pnl_std > 0 else 0.0

print(round(sharpe(backtest(synthetic_prices())), 3))
